# Knowledge-Gap Dataset QA Notebook

Runs a full quality check on `data/processed/train.csv` (and `eval.csv` for the leakage check),
covering the same checks used to verify the augmented-window dataset before it was committed:

1. Structural integrity (shape, columns, missing values, label values)
2. Augmented window technique (variant sizes, dynamic sizing, anchor content)
3. Sentence separation quality (no glued headings, no unrepaired sentence boundaries)
4. Duplicates
5. Train/eval leakage
6. Label balance and window length distribution

**Before running**: launch Jupyter from the repo root (`cd` into the cloned `thesis` folder, then `jupyter notebook`),
so the relative paths below resolve correctly. Run cells top to bottom.

In [ ]:
import sys, os, re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# sanity check: are we running from the repo root?
assert os.path.exists('data/processed/train.csv'), (
    "Can't find data/processed/train.csv from the current directory.\n"
    "Launch Jupyter from the repo root (the folder containing build_dataset.py), not a subfolder."
)

sys.path.insert(0, 'src')

results = []  # (check_name, passed: bool, detail: str)
def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(("PASS" if ok else "FAIL"), "-", name, ("::" + detail if detail else ""))

In [ ]:
train = pd.read_csv('data/processed/train.csv')
eval_ = pd.read_csv('data/processed/eval.csv')
print('train:', train.shape)
print('eval: ', eval_.shape)
train.head()

## 1. Structural integrity

In [ ]:
expected_cols = {'article_id','section','text','label','source','variant','promoted','anchor_text','n_sentences'}
check('train.csv has expected columns', set(train.columns) == expected_cols, str(set(train.columns)))
check('eval.csv has expected columns', set(eval_.columns) == expected_cols, str(set(eval_.columns)))

required = ['article_id','section','text','label']
missing_t = train[required].isna().any(axis=1).sum()
missing_e = eval_[required].isna().any(axis=1).sum()
check('no missing required fields (train)', missing_t == 0, f'{missing_t} bad rows')
check('no missing required fields (eval)', missing_e == 0, f'{missing_e} bad rows')

bad_labels_t = (~train['label'].isin([0,1])).sum()
bad_labels_e = (~eval_['label'].isin([0,1])).sum()
check('labels are only 0/1 (train)', bad_labels_t == 0, f'{bad_labels_t} bad')
check('labels are only 0/1 (eval)', bad_labels_e == 0, f'{bad_labels_e} bad')

## 2. Augmented window technique

Checks that the dynamic-window design was actually applied: anchor-centered windows should vary in size
(anchor length + 2, not a fixed size), nearby-negative blocks should always be 1-3 sentences, and every
anchor's text should be recoverable from its window.

**Note on the anchor-content check**: a small number of rows will legitimately fail a *strict* byte-exact
substring check. This happens when the raw annotated `anchor_text` itself contains an extraction artifact
(a glued sentence boundary, a stray heading word like `"Methods"`, or a multi-number citation marker like
`"disease.44,45 However"`) that the pipeline correctly repairs before building the window text. That's the
pipeline working as intended, not a defect — the cell below checks for *content* loss (all real words
present, ignoring whitespace/digit/comma noise) rather than requiring an exact character match, and prints
any row that fails even that looser check for you to inspect.

In [ ]:
variant_counts = (train['source'] + '/' + train['variant']).value_counts()
print(variant_counts)

expected_anchor_variants = {'2before_0after','1before_1after','0before_2after'}
expected_nearby_variants = {'before_block','after_block'}
anchor_srcs = {'positive_anchor','negative_anchor'}

bad_variant = train[
    (train['source'].isin(anchor_srcs) & ~train['variant'].isin(expected_anchor_variants)) |
    ((train['source']=='nearby_negative') & ~train['variant'].isin(expected_nearby_variants))
]
check('all variant labels match design', len(bad_variant)==0, f'{len(bad_variant)} unexpected rows')

nb = train[train['source']=='nearby_negative']
bad_nb_size = nb[~nb['n_sentences'].between(1,3)]
check('nearby_negative windows are 1-3 sentences', len(bad_nb_size)==0,
      f'{len(bad_nb_size)} out of range; sizes={dict(nb["n_sentences"].value_counts())}')

anchor_rows = train[train['source'].isin(anchor_srcs)]
n_unique_sizes = anchor_rows['n_sentences'].nunique()
check('anchor-centered window sizes are dynamic, not fixed', n_unique_sizes > 1,
      f'{n_unique_sizes} distinct sizes: {sorted(anchor_rows["n_sentences"].unique())}')

In [ ]:
def strip_noise(s):
    # whitespace AND digit/comma runs -- the pipeline also strips stray
    # multi-number citation markers (e.g. "disease.44,45 However"), so a
    # fair content-preservation check has to look past those too, not
    # just whitespace normalization.
    return re.sub(r'[\d,\s]+', '', str(s))

anchor_rows = train[train['source'].isin(anchor_srcs)]
byte_exact_fail = anchor_rows[~anchor_rows.apply(lambda r: r['anchor_text'] in r['text'], axis=1)]
content_fail = byte_exact_fail[~byte_exact_fail.apply(
    lambda r: strip_noise(r['anchor_text']) in strip_noise(r['text']), axis=1
)]

print(f"{len(byte_exact_fail)} rows fail a strict byte-exact substring check")
print(f"{len(content_fail)} rows still fail after stripping whitespace/digits/commas (= potential real content issues)")
check('no real anchor-content loss (noise-tolerant check)', len(content_fail)==0, f'{len(content_fail)} rows')

if len(content_fail) > 0:
    display(content_fail[['article_id','section','variant','anchor_text','text']].head(10))

## 3. Sentence separation quality

In [ ]:
heading_words = ['Background','Objective','Objectives','Purpose','Aim','Aims','Abstract',
                  'Methods?','Results','Discussion','Discussions','Conclusions?','Limitations',
                  'Introduction','Rationale','Interpretation','Diagnoses','Diagnosis',
                  'Intervention','Interventions','Outcomes?','Lessons']
heading_re = re.compile(r'(?<![A-Za-z])(?:' + '|'.join(heading_words) + r')(?::)?(?=[A-Z])')
bad_heading = train[train['text'].str.contains(heading_re.pattern, regex=True)]
check('no leftover glued structural headings in window text', len(bad_heading)==0, f'{len(bad_heading)} rows')
if len(bad_heading) > 0:
    display(bad_heading[['article_id','text']].head(5))

In [ ]:
try:
    from pipeline.segmentation import _HGVS_RE, _ABBREV_RE
    have_module = True
except ImportError as e:
    have_module = False
    print(f'Could not import pipeline.segmentation ({e}) -- run `pip install -r requirements.txt` first.')
    print('Falling back to a simpler check without HGVS-notation protection built in.')

glue_re = re.compile(r'[A-Za-z0-9]\.(?=[A-Z])')
unrepaired = []
for _, row in train.iterrows():
    text = row['text']
    for m in glue_re.finditer(text):
        period_pos = m.start() + 1
        protected = False
        if have_module:
            protected = (
                any(pm.end()-1 == period_pos for pm in _HGVS_RE.finditer(text)) or
                any(pm.end()-1 == period_pos for pm in _ABBREV_RE.finditer(text))
            )
        if not protected:
            unrepaired.append((row['article_id'], text[max(0,m.start()-30):m.start()+30]))

check('no unrepaired glued sentence boundaries', len(unrepaired)==0,
      f'{len(unrepaired)} occurrences' + (f'; e.g. {unrepaired[0]}' if unrepaired else ''))

## 4. Duplicates

In [ ]:
dup_groups = train.groupby(['article_id','text']).size()
dupes = dup_groups[dup_groups > 1]
check('no duplicate (article_id, text) pairs', len(dupes)==0, f'{len(dupes)} dup groups')

label_variety = train.groupby(['article_id','text'])['label'].nunique()
conflicts = (label_variety > 1).sum()
check('no same-text rows with conflicting labels', conflicts==0, f'{conflicts} conflicts')

## 5. Train/eval leakage

In [ ]:
train_ids = set(train['article_id'])
eval_ids = set(eval_['article_id'])
overlap = train_ids & eval_ids
check('zero article_id overlap between train and eval', len(overlap)==0,
      f'{len(overlap)} shared: {list(overlap)[:5]}')

## 6. Label balance and window length

In [ ]:
for name, df in [('train', train), ('eval', eval_)]:
    n_pos = (df['label']==1).sum()
    n_neg = (df['label']==0).sum()
    print(f'{name}: {len(df)} total, {n_pos} positive ({n_pos/len(df):.1%}), {n_neg} negative ({n_neg/len(df):.1%})')

In [ ]:
word_counts = train['text'].str.split().str.len()
print(word_counts.describe(percentiles=[.5,.9,.95,.99]))

plt.figure(figsize=(8,4))
plt.hist(word_counts, bins=40)
plt.xlabel('Words per window')
plt.ylabel('Count')
plt.title('Window length distribution (train.csv)')
plt.show()

## Summary

In [ ]:
n_pass = sum(1 for _,ok,_ in results if ok)
n_fail = sum(1 for _,ok,_ in results if not ok)
print(f'{n_pass} passed, {n_fail} failed out of {len(results)} checks\n')
for name, ok, detail in results:
    print(('PASS' if ok else 'FAIL'), '-', name, ('::' + detail if detail else ''))